# Phase 1 & 2 Visual Audit: Ingestion, Standardization & Tagging

This thin verification notebook imports directly from `src/` to visually audit:
1. **Base-36 ID Preservation:** Verifies that Reddit IDs starting with `t` or `3` are never truncated.
2. **Atomic Text Separation:** Confirms post text is on `depth=0` and comment rows (`depth>=1`) have 0% post title contamination.
3. **Canonical Parquet Schema:** Checks data types, UTC timestamps, and round-trip serialization.
4. **Entity Disambiguation:** Checks regex negative pattern suppression (e.g. 'Monthly Meta-Thread') and multi-company bridge generation.
5. **Workplace Relevance:** Inspects filtering precision, drop reasons, and lexicon matching.
6. **Two-Stage Deduplication:** Verifies viral repost collapse into unique text rows with `duplicate_count` tracking.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on Python sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
from src.common.config import EngineConfig
from src.common.azure_storage import StorageManager
from src.common.text_utils import clean_reddit_id, compute_content_hash, normalize_text
from src.ingest.standardizer import build_canonical_record, to_canonical_dataframe
from src.ingest.validator import validate_canonical_records
from src.tagging.entity_disambiguator import EntityDisambiguator
from src.tagging.relevance_filter import RelevanceFilter
from src.tagging.deduplicator import TwoStageDeduplicator
from src.tagging.pipeline import TaggingPipeline

config = EngineConfig.load()
storage = StorageManager(config)
print(f"✅ Engine configured. Project root: {config.project_root}")
print(f"Target companies loaded: {len(config.companies.get('companies', {}))}")
print(f"Target subreddits loaded: {len(config.subreddits.get('subreddits', {}))}")

### 1. Base-36 ID Cleanliness Check

In [ ]:
test_cases = ["t3_1oxu8rh", "t1_abcdef", "t3_3abcde", "t1_t12345", "3abcde"]
print("Raw ID -> Cleaned ID (Must preserve leading '3' or 't'):")
for raw in test_cases:
    clean = clean_reddit_id(raw)
    print(f"  {raw:15s} -> {clean}")
    assert not clean.startswith("t3_") and not clean.startswith("t1_")
print("\nAll ID cleaning checks passed!")

### 2. Standardization & Atomic Text Separation Audit

In [ ]:
# Create synthetic batch representing post, child comment, and viral repost
sample_records = [
    build_canonical_record(
        post_id="t3_1oxu8rh",
        depth=0,
        record_type="post",
        subreddit="cscareerquestions",
        author="software_lead",
        created_utc="2026-01-15T10:00:00Z",
        title="Andy Jassy announces 5-day RTO mandate effective immediately",
        body="Amazon leadership stated this will foster collaboration and culture.",
        score=340,
        num_comments=85,
        source="audit_sample"
    ),
    build_canonical_record(
        post_id="t3_1oxu8rh",
        comment_id="t1_c001",
        parent_id="t3_1oxu8rh",
        root_comment_id="t1_c001",
        depth=1,
        record_type="comment",
        subreddit="cscareerquestions",
        author="senior_eng",
        created_utc="2026-01-15T10:30:00Z",
        title="", # Must be empty
        body="Morale in our org just tanked. Half the team is updating resumes.",
        score=120,
        source="audit_sample"
    ),
    build_canonical_record(
        post_id="t3_meta_noise",
        depth=0,
        record_type="post",
        subreddit="cscareerquestions",
        author="mod",
        created_utc="2026-01-01T00:00:00Z",
        title="Monthly Meta-Thread for Career Advice and Feedback",
        body="Please post all general resume advice and questions here.",
        score=15,
        source="audit_sample"
    ),
    build_canonical_record(
        post_id="t3_repost_1",
        depth=0,
        record_type="post",
        subreddit="antiwork",
        author="viral_poster",
        created_utc="2026-01-16T12:00:00Z",
        title="Amazon VP Resigns Over Warehouse Conditions",
        body="I am resigning in protest of corporate treatment of warehouse workers.",
        score=4500,
        source="audit_sample"
    ),
    build_canonical_record(
        post_id="t3_repost_2",
        depth=0,
        record_type="post",
        subreddit="jobs",
        author="crosspost_bot",
        created_utc="2026-01-16T13:00:00Z",
        title="Amazon VP Resigns Over Warehouse Conditions",
        body="I am resigning in protest of corporate treatment of warehouse workers.",
        score=200,
        source="audit_sample"
    ),
]

df_sample = to_canonical_dataframe(sample_records)
is_valid, errors = validate_canonical_records(df_sample)
print(f"Validation Passed: {is_valid}")
if errors:
    print("Errors:", errors)

# Verify zero title leakage on comments
comment_titles = df_sample[df_sample["depth"] >= 1]["title"]
leak_count = (comment_titles != "").sum()
print(f"Comment Title Leakage Count: {leak_count} (Expected: 0)")
df_sample[["record_id", "post_id", "comment_id", "depth", "title", "clean_text", "year_month"]]

### 3. Entity Disambiguation Audit

In [ ]:
disambiguator = EntityDisambiguator(config)
df_tagged, df_mentions = disambiguator.tag_dataframe(df_sample)

print("Tagged Records Overview:")
for _, row in df_tagged.iterrows():
    print(f"  [{row['record_id']}] Title: {row['title'][:40]:<40} -> Companies: {row['companies']}")

# Verify 'Monthly Meta-Thread' was NOT tagged as Meta
meta_noise_row = df_tagged[df_tagged["record_id"] == "p_meta_noise"].iloc[0]
print(f"\n'Monthly Meta-Thread' tagged as: {meta_noise_row['companies']} (Expected: [])")
assert "Meta" not in meta_noise_row["companies"], "Monthly Meta-Thread false positive caught!"

print("\nCompany Mentions Bridge Table:")
display(df_mentions[["record_id", "company", "alias", "match_source", "confidence"]])

### 4. Workplace Relevance Filtering Audit

In [ ]:
filter_mod = RelevanceFilter(config)
df_relevant, df_dropped = filter_mod.filter_dataframe(df_tagged)

print(f"Relevant Count: {len(df_relevant)} / {len(df_tagged)}")
print(f"Dropped Count:  {len(df_dropped)} / {len(df_tagged)}")
if len(df_dropped) > 0:
    print("\nDropped Records Reasons:")
    for _, r in df_dropped.iterrows():
        print(f"  [{r['record_id']}] Reason: {r['drop_reason']}")

display(df_relevant[["record_id", "subreddit", "relevance_score", "matched_work_terms", "companies"]])

### 5. Two-Stage Deduplication Audit

In [ ]:
deduplicator = TwoStageDeduplicator()
df_deduped, df_clean_mentions = deduplicator.deduplicate(df_relevant, df_mentions)

print(f"Relevant records before dedup: {len(df_relevant)}")
print(f"Unique records after dedup:   {len(df_deduped)}")
print("\nSurviving Unique Records with Duplicate Counts:")
display(df_deduped[["record_id", "title", "duplicate_count", "score", "companies"]])

# Check viral repost duplicate_count
viral_row = df_deduped[df_deduped["title"].str.contains("VP Resigns", na=False)].iloc[0]
print(f"Viral post duplicate count: {viral_row['duplicate_count']} (Expected: 2)")
assert viral_row["duplicate_count"] == 2

### 6. Parquet Storage Verification

In [ ]:
saved_rel = storage.save_parquet(df_deduped, "standardized/relevant_records_audit.parquet")
saved_men = storage.save_parquet(df_clean_mentions, "standardized/company_mentions_audit.parquet")
print(f"✅ Successfully saved relevant records to: {saved_rel}")
print(f"✅ Successfully saved company mentions bridge table to: {saved_men}")

reloaded = storage.load_parquet("standardized/relevant_records_audit.parquet")
print(f"Reloaded {len(reloaded)} rows with columns:", list(reloaded.columns))